# Phase 2 experiments on Google Colab or Kaggle

Runs every Phase 2 experiment (E1-E6) in order: a quick smoke test, the validation runs, the
second-round trigger choice (validation split only, rule fixed in advance in `docs/devlog.md`),
then the test runs. Every reply is cached and every run resumes, so you can stop at any time and
continue later, even on the other platform.

**Before you start**
- **Colab:** `Runtime > Change runtime type > T4 GPU`. In the **Secrets** panel (key icon) add
  `NCBI_EMAIL` (and optionally `NCBI_API_KEY`) and switch on "Notebook access".
- **Kaggle:** in the notebook settings turn on **GPU** and **Internet** (Internet needs a
  phone-verified account). Add `NCBI_EMAIL` (and optionally `NCBI_API_KEY`) under
  `Add-ons > Secrets`.

Run the cells in order. The reply cache is saved after each run; if a session drops mid-run,
finished items are kept and the run resumes. After a disconnect, run the cells again from the top.

In [ ]:
import os
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/repo"
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "other")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

Clones the repository the first time and pulls the latest version after that.

In [ ]:
REPO_URL = "https://github.com/khan-shayan9/Agentic-AI-Project.git"
if not os.path.exists(REPO):
    !git clone -q $REPO_URL $REPO
else:
    !git -C $REPO pull -q
%cd $REPO
!pip install -q -e .

## 2. Keep work between sessions

- **Colab:** the run folders (`results/runs`) live on Google Drive in `MyDrive/medverify_phase2`.
  The reply cache stays on local disk (SQLite is not safe on Drive); it is copied to
  `MyDrive/medverify_phase2/state` after every run and restored from there here.
- **Kaggle:** attach the dataset made from `medverify_state.zip` (last cell) and it is restored here.

The chosen trigger (`experiments/final_config.json`) is saved and restored with the cache.

In [ ]:
import glob, shutil, zipfile
from pathlib import Path

BACKUP = None  # where the cache backup lives (none when running elsewhere)
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = Path("/content/drive/MyDrive/medverify_phase2")
    BACKUP = STORE / "state"
    runs, link = STORE / "results/runs", Path(REPO) / "results/runs"
    runs.mkdir(parents=True, exist_ok=True)
    if not link.is_symlink():
        if link.exists():  # move anything made before this cell onto Drive first
            shutil.copytree(link, runs, dirs_exist_ok=True)
            shutil.rmtree(link)
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(runs, target_is_directory=True)
    print("Run folders are on Drive:", runs)
elif ON_KAGGLE:
    BACKUP = Path("/kaggle/working/medverify_state")
    zips = sorted(glob.glob("/kaggle/input/*/medverify_state.zip"))
    folders = sorted(glob.glob("/kaggle/input/*/medverify_state"))  # Kaggle may unzip an uploaded archive itself
    root = None
    if zips:
        root = Path("/tmp/medverify_restore")
        zipfile.ZipFile(zips[0]).extractall(root)
    elif folders:
        root = Path(folders[0]).parent
    if root is not None:
        if (root / "results/runs").is_dir():
            shutil.copytree(root / "results/runs", Path(REPO) / "results/runs", dirs_exist_ok=True)
        if (root / "medverify_state").is_dir():
            shutil.copytree(root / "medverify_state", BACKUP, dirs_exist_ok=True)
        print("Unpacked the attached state from", zips[0] if zips else root)
    else:
        print("No saved state attached; starting fresh.")

if BACKUP is not None and BACKUP.is_dir():
    !python -m experiments.state restore --from "{BACKUP}"
backup_arg = f'--backup-dir "{BACKUP}"' if BACKUP is not None else ""

## 3. Install Ollama and the three local models

Ollama runs with a 4096-token context (`OLLAMA_CONTEXT_LENGTH=4096`). If the session restarts,
run the second cell again: it restarts the server (models already downloaded are not downloaded
again). The exact model versions are saved to `results/runs/ollama_versions.txt`.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests

ollama = subprocess.Popen(["ollama", "serve"], env=dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096"),
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start within a minute")

for model in ["llama3.1:8b", "mistral:7b", "phi4-mini"]:
    !ollama pull {model}

versions = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
Path("results/runs").mkdir(parents=True, exist_ok=True)
Path("results/runs/ollama_versions.txt").write_text(versions)
print(versions)

## 4. Load your keys

`NCBI_EMAIL` is needed for PubMed; `NCBI_API_KEY` is optional (it raises the PubMed limit from 3
to 10 requests per second).

In [ ]:
def secret(name):
    try:
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception:
        return None

loaded = []
for name in ["NCBI_EMAIL", "NCBI_API_KEY"]:
    value = os.environ.get(name) or secret(name)
    if value:
        os.environ[name] = value
        loaded.append(name)
assert "NCBI_EMAIL" in loaded, "Add NCBI_EMAIL as a secret (see the first cell), then run this cell again."
print("Loaded:", loaded)

## 5. Prepare the data

Downloads MedHallu and makes the splits, then checks that they are the same as in Phase 1.

In [ ]:
import json

!python -m medverify prepare
now = json.loads(Path("data/processed/split_summary.json").read_text())
phase1 = json.loads(Path("results/phase1/split_summary.json").read_text())
assert now["seed"] == phase1["seed"] and now["splits"] == phase1["splits"], "The splits differ from Phase 1: stop here."
print("Splits match Phase 1:", now["splits"])

## 6. Smoke test

Every validation and test run on 6 dev answers, to catch a crash before spending GPU time.

In [ ]:
!python -m experiments.run --stage smoke {backup_arg}

## 7. Validation runs and the trigger choice

The three validation runs (second round off, `split`, `no-majority`) on the 200 validation
answers. This takes a few hours on a T4; if the session stops, run the cell again and it continues.
Then `experiments.choose` applies the rule fixed in advance and writes
`experiments/final_config.json`, which is saved with the cache backup. Download that file and
commit it to the repository.

In [ ]:
!python -m experiments.run --stage val {backup_arg}

In [ ]:
!python -m experiments.choose
if BACKUP is not None:
    !python -m experiments.state save --to "{BACKUP}"

## 8. Test runs

Set `ONLY` to one experiment at a time (`E1`, `E2`, `E3` or `E5`) to split the work across
sessions, or between Colab and Kaggle; `""` runs everything. The first cell shows what is left.

In [ ]:
!python -m experiments.run --stage test --dry-run

In [ ]:
ONLY = "E2"
only_arg = f"--only {ONLY}" if ONLY else ""
!python -m experiments.run --stage test {only_arg} {backup_arg}

## 9. Save

Saves the cache once more. On Colab the run folders and the cache backup are already on Drive.
On Kaggle this zips `results/runs` and the `medverify_state` backup folder into
`/kaggle/working/medverify_state.zip`: download it, or save the notebook version and attach its
output as a dataset in the next session.

In [ ]:
if BACKUP is not None:
    !python -m experiments.state save --to "{BACKUP}"
if ON_KAGGLE:
    out = Path("/kaggle/working/medverify_state.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for path in Path("results/runs").rglob("*"):
            if path.is_file():
                z.write(path)
        for path in BACKUP.rglob("*"):
            if path.is_file():
                z.write(path, Path("medverify_state") / path.relative_to(BACKUP))
    print(f"Saved {out} ({out.stat().st_size / 1e6:.1f} MB)")
elif ON_COLAB:
    print("The run folders and the cache backup are on Drive (MyDrive/medverify_phase2).")